![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 17 -- Lab 1: RNN Text Classification

**Scenario:** A news agency receives thousands of articles every day and needs to automatically sort them into categories: World, Sports, Business, or Science/Technology. Your job is to build three different neural networks -- a vanilla **RNN**, an **LSTM**, and a **GRU** -- to classify news headlines, then compare which one works best.

This connects directly to the **many-to-one** RNN architecture from today's slides: read a sequence of words, produce a single classification.

| Part | Goal |
|---|---|
| Part 1 | Load the AG News dataset and build a vocabulary |
| Part 2 | Convert text to numbers and create DataLoaders |
| Part 3 | Build and train a vanilla RNN classifier |
| Part 4 | Build and train an LSTM classifier |
| Part 5 | Build and train a GRU classifier |
| Part 6 | Compare all three models |

---

## Setup

In [ ]:
!pip install torchtext torch -q

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import re
import time

plt.rcParams['figure.dpi'] = 120

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

---

## Part 1: Load and Explore the Data

We will use the **AG News** dataset -- 120,000 training and 7,600 test news articles in 4 categories.

We download the CSV files directly from a public mirror.

In [ ]:
# --- GIVEN ---
import csv
import os
import urllib.request

def download_ag_news():
    """Download AG News CSV files."""
    base_url = "https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/"
    os.makedirs('data', exist_ok=True)
    for split in ['train', 'test']:
        path = f'data/{split}.csv'
        if not os.path.exists(path):
            print(f'Downloading {split}.csv...')
            urllib.request.urlretrieve(base_url + f'{split}.csv', path)
    return 'data/train.csv', 'data/test.csv'

def load_ag_news_csv(filepath):
    """Load AG News CSV: each row is (class, title, description)."""
    labels, texts = [], []
    with open(filepath, 'r', encoding='utf-8') as f:
        reader = csv.reader(f)
        for row in reader:
            label = int(row[0]) - 1
            text = row[1] + ' ' + row[2]
            labels.append(label)
            texts.append(text)
    return texts, labels

train_csv, test_csv = download_ag_news()
train_texts, train_labels = load_ag_news_csv(train_csv)
test_texts, test_labels = load_ag_news_csv(test_csv)

class_names = ['World', 'Sports', 'Business', 'Sci/Tech']

print(f'Training samples: {len(train_texts)}')
print(f'Test samples:     {len(test_texts)}')
print(f'Classes: {class_names}')
print(f'\nExample:')
print(f'  Label: {class_names[train_labels[0]]}')
print(f'  Text:  {train_texts[0][:150]}...')

### Task 1: Explore the Class Distribution

Count how many samples are in each class and create a bar chart.

**TODO:**
- Use `Counter(train_labels)` to count class frequencies
- Create a bar chart with `plt.bar(class_names, ...)`
- Print the count for each class

In [ ]:
# Your code here


---

## Part 2: Text to Numbers

Neural networks need numbers, not words. We need to:
1. **Tokenize** -- split text into words
2. **Build a vocabulary** -- assign each word a unique integer
3. **Encode** -- convert each text to a list of integers
4. **Pad/truncate** -- make all sequences the same length

### Step 1: Tokenizer and Vocabulary (GIVEN)

Run the cell below. It builds a simple word tokenizer and a vocabulary from the training data.

In [ ]:
# --- GIVEN ---
def simple_tokenizer(text):
    """Lowercase and split on non-alphanumeric characters."""
    return re.findall(r'\w+', text.lower())

# Build vocabulary from training data
MAX_VOCAB = 25000
word_counts = Counter()
for text in train_texts:
    word_counts.update(simple_tokenizer(text))

# Reserve 0 for <pad> and 1 for <unk>
vocab = {'<pad>': 0, '<unk>': 1}
for word, _ in word_counts.most_common(MAX_VOCAB - 2):
    vocab[word] = len(vocab)

VOCAB_SIZE = len(vocab)
PAD_IDX = vocab['<pad>']
print(f'Vocabulary size: {VOCAB_SIZE}')
print(f'Sample words: {list(vocab.items())[2:12]}')

### Task 2: Encode Text and Create a Dataset

Convert each text into a list of vocabulary indices. Then wrap everything in a PyTorch `Dataset`.

**TODO:**
- Complete `encode_text()`: tokenize the text, look up each token in `vocab` (use `vocab['<unk>']` for unknown words), truncate to `MAX_LEN`
- Complete `NewsDataset`: store encoded texts and labels, return them in `__getitem__`
- Complete `collate_fn`: pad all sequences in a batch to the same length
- Create train and test DataLoaders with `batch_size=64`

In [ ]:
MAX_LEN = 128

def encode_text(text):
    """Tokenize and convert to vocabulary indices."""
    # Your code here
    pass

class NewsDataset(Dataset):
    def __init__(self, texts, labels):
        # Your code here
        pass

    def __len__(self):
        # Your code here
        pass

    def __getitem__(self, idx):
        # Your code here
        pass

def collate_fn(batch):
    """Pad sequences to the same length within a batch."""
    # Your code here
    pass

# Create datasets and dataloaders
BATCH_SIZE = 64
# Your code here


---

## Part 3: Build and Train a Vanilla RNN

Our first model is a simple RNN: Embedding -> RNN -> Linear.

This is the **many-to-one** architecture from the slides: read a sequence of words, use the **last hidden state** to classify.

### Task 3: Define the RNN Model

**TODO:** Build a model with:
- `nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)`
- `nn.RNN(embed_dim, hidden_dim, batch_first=True)`
- `nn.Linear(hidden_dim, num_classes)`

In `forward()`: embed the input, pass through the RNN, take the last hidden state `h_n`, and classify it.

In [ ]:
class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, pad_idx):
        super().__init__()
        # Your code here

    def forward(self, x, lengths):
        # Your code here
        pass

EMBED_DIM = 64
HIDDEN_DIM = 128
NUM_CLASSES = 4

rnn_model = RNNClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, NUM_CLASSES, PAD_IDX).to(device)
print(rnn_model)
total_params = sum(p.numel() for p in rnn_model.parameters())
print(f'Total parameters: {total_params:,}')

### Training and Evaluation Functions (GIVEN)

We define reusable training and evaluation functions so we can use them for all three models.

In [ ]:
# --- GIVEN ---
def train_one_epoch(model, dataloader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for texts, labels, lengths in dataloader:
        texts, labels = texts.to(device), labels.to(device)
        lengths = lengths.to(device)

        logits = model(texts, lengths)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        predictions = logits.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, 100 * correct / total

def evaluate(model, dataloader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for texts, labels, lengths in dataloader:
            texts, labels = texts.to(device), labels.to(device)
            lengths = lengths.to(device)
            logits = model(texts, lengths)
            predictions = logits.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
    return 100 * correct / total

def train_model(model, num_epochs=5, lr=1e-3):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    history = {'train_loss': [], 'train_acc': [], 'test_acc': []}

    start = time.time()
    for epoch in range(1, num_epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        test_acc = evaluate(model, test_loader)
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_acc'].append(test_acc)
        print(f'Epoch {epoch}/{num_epochs}  |  Loss: {train_loss:.4f}  |  Train Acc: {train_acc:.1f}%  |  Test Acc: {test_acc:.1f}%')
    elapsed = time.time() - start
    print(f'Training took {elapsed:.1f}s')
    return history

### Task 4: Train the RNN

In [ ]:
NUM_EPOCHS = 5
print('Training Vanilla RNN...')
print('=' * 75)
rnn_history = train_model(rnn_model, num_epochs=NUM_EPOCHS)

### Task 5: Show Predictions

Pick 10 random test samples and show the model's prediction vs the true label.

**TODO:**
- Pick 10 random indices from the test set
- For each: encode the text, run it through the model, compare predicted vs true label

In [ ]:
def show_predictions(model, n=10):
    """Show n random predictions from the test set."""
    # Your code here
    pass

print('\nVanilla RNN Predictions:')
show_predictions(rnn_model)

---

## Part 4: Build and Train an LSTM Classifier

The LSTM uses **gates** (forget, input, output) and a **cell state** to remember long-range dependencies. The code change is minimal -- we swap `nn.RNN` for `nn.LSTM`.

### Task 6: Define the LSTM Model

**TODO:** Copy the RNN model and make two changes:
- Use `nn.LSTM` instead of `nn.RNN`
- In forward, LSTM returns `(output, (h_n, c_n))` -- unpack both `h_n` and `c_n`

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, pad_idx):
        super().__init__()
        # Your code here

    def forward(self, x, lengths):
        # Your code here
        pass

lstm_model = LSTMClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, NUM_CLASSES, PAD_IDX).to(device)
print(lstm_model)
total_params = sum(p.numel() for p in lstm_model.parameters())
print(f'Total parameters: {total_params:,}')

### Task 7: Train the LSTM

In [ ]:
print('Training LSTM...')
print('=' * 75)
lstm_history = train_model(lstm_model, num_epochs=NUM_EPOCHS)

In [ ]:
print('\nLSTM Predictions:')
show_predictions(lstm_model)

---

## Part 5: Build and Train a GRU Classifier

The GRU is a simpler alternative to LSTM with only 2 gates (reset, update) instead of 3. It's faster to train and often performs similarly.

### Task 8: Define the GRU Model

**TODO:** Same as LSTM, but use `nn.GRU`. Note that GRU returns `(output, h_n)` -- no cell state, just like the vanilla RNN.

In [ ]:
class GRUClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, pad_idx):
        super().__init__()
        # Your code here

    def forward(self, x, lengths):
        # Your code here
        pass

gru_model = GRUClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, NUM_CLASSES, PAD_IDX).to(device)
print(gru_model)
total_params = sum(p.numel() for p in gru_model.parameters())
print(f'Total parameters: {total_params:,}')

### Task 9: Train the GRU

In [ ]:
print('Training GRU...')
print('=' * 75)
gru_history = train_model(gru_model, num_epochs=NUM_EPOCHS)

In [ ]:
print('\nGRU Predictions:')
show_predictions(gru_model)

---

## Part 6: Compare All Three Models

### Task 10: Accuracy Comparison

**TODO:**
- Get the final test accuracy from each model's history
- Create a bar chart comparing the three models
- Plot the training loss curves on a second subplot
- Print a comparison table (model name, parameters, test accuracy)

In [ ]:
# Your code here


---

## Discussion

1. Which model achieved the highest accuracy? Were you surprised?
2. Look at the parameter counts. How many more parameters does LSTM have compared to the vanilla RNN? Why?
3. In the side-by-side predictions, are there examples where one model is correct and the others are wrong? What might cause this?

---

## Wrap-Up

**What you learned:**
- How to convert raw text into numerical sequences for a neural network
- How to build a **vanilla RNN**, **LSTM**, and **GRU** classifier in PyTorch
- The code difference between them is minimal (`nn.RNN` vs `nn.LSTM` vs `nn.GRU`)
- LSTM and GRU generally outperform vanilla RNN thanks to their gating mechanisms
- LSTM has more parameters (3 gates + cell state) than GRU (2 gates), but GRU is often comparable in accuracy

**Next:** Lab 2 -- Build Your Own Mini-GPT!